# baseline v3

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

윈도우 데스크탑의 RTX 5060 ti GPU 환경에서 개발되었습니다.

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- ipykernel 설치
- 아래 셀 다시 실행 : 무한 로딩 시 restart
- hello 출력시 torch 설치

In [1]:
print('hello123')

hello123


In [2]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

Looking in indexes: https://download.pytorch.org/whl/nightly/cu128



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

2.12.0.dev20260401+cu128
True
NVIDIA GeForce RTX 5060 Ti


In [4]:
!pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade 


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

# 라이브러리, 데이터, 설정

In [5]:
import os, re, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Dict, List, Any
from transformers import (
    AutoModelForVision2Seq,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

# 이미지 로드 시 픽셀 제한 해제
Image.MAX_IMAGE_PIXELS = None

# 디바이스 GPU 우선 사용 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# 사전 학습 모델 정의
MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
IMAGE_SIZE = 384
MAX_NEW_TOKENS = 8
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# 데이터셋 로드
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

# 학습데이터 200개만 추출 TODO: 수정
train_df = train_df.sample(n=200, random_state=SEED).reset_index(drop=True)

c:\SSAFY\2026-ssafy-ai-15-2\baseline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [ ]:
# 양자화
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# 프로세서
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256*32*32,
    max_pixels=1280*32*32,
    trust_remote_code=True,
)

# 사전학습 모델
base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# 양자화 모델로 로드
base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

# LoRA 세팅
lora_config = LoraConfig(
     r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

# PEFT 모델 생성
model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

c:\SSAFY\2026-ssafy-ai-15-2\baseline\Lib\site-packages\transformers\models\auto\modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
W0402 11:33:41.362000 8860 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [ ]:
# 모델 지시사항
SYSTEM_INSTRUCT = (
    "You are a helpful visual question answering assistant. "
    "Answer using exactly one letter among a, b, c, or d. No explanation."
)

# 프롬프트
def build_mc_prompt(question, a, b, c, d):
    return (
        f"{question}\n"
        f"(a) {a}\n(b) {b}\n(c) {c}\n(d) {d}\n\n"
        "Answer with one letter only (a, b, c, or d):"
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [ ]:
# 커스텀 데이터셋
class VQAMCDataset(Dataset):
    def __init__(self, df, processor, train=True):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.train = train

    def __len__(self): return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row["path"]).convert("RGB")

        q = str(row["question"])
        a, b, c, d = str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        user_text = build_mc_prompt(q, a, b, c, d)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":user_text}
            ]}
        ]
        if self.train:
            gold = str(row["answer"]).strip().lower()
            messages.append({"role":"assistant","content":[{"type":"text","text":gold}]})

        return {"messages": messages, "image": img}

# ── DataCollator 수정 ─────────────────────────────────
@dataclass
class DataCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        texts, images = [], []
        for sample in batch:
            text = self.processor.apply_chat_template(
                sample["messages"],
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
            images.append(sample["image"])

        enc = self.processor(
            text=texts, images=images,
            padding=True, return_tensors="pt"
        )

        if self.train:
            labels = enc["input_ids"].clone()

            for idx, sample in enumerate(batch):
                # processor 재호출 없이 텍스트 길이만으로 prompt_len 계산
                prompt_only = self.processor.apply_chat_template(
                    sample["messages"][:-1],
                    tokenize=False,
                    add_generation_prompt=True
                )
                # 이미지 없이 텍스트만 토크나이징 → 훨씬 빠름
                prompt_len = len(self.processor.tokenizer(
                    prompt_only, return_tensors="pt"
                )["input_ids"][0])

                labels[idx, :prompt_len] = -100

            labels[labels == self.processor.tokenizer.pad_token_id] = -100
            enc["labels"] = labels

        return enc


# ── DataLoader 수정 ───────────────────────────────────
train_loader = DataLoader(
    train_ds, batch_size=1, shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=2,          # ← 0에서 변경
    pin_memory=True,        # ← 추가
    persistent_workers=True # ← worker 재시작 오버헤드 제거
)
valid_loader = DataLoader(
    valid_ds, batch_size=1, shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=2,
    pin_memory=True,
    persistent_workers=True
)


# ── Processor 이미지 크기 축소 ─────────────────────────
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=256*28*28,   # ← 32*32 → 28*28로 축소
    max_pixels=640*28*28,   # ← 1280 → 640으로 절반 축소 (속도 2~3배 향상)
    trust_remote_code=True,
)

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [ ]:
# 검증용 데이터 분리
split = int(len(train_df) * 0.9)
train_subset, valid_subset = train_df.iloc[:split], train_df.iloc[split:]

# VQAMCDataset 형태로 변환
train_ds = VQAMCDataset(train_subset, processor, train=True)
valid_ds = VQAMCDataset(valid_subset, processor, train=True)

# 데이터로더 (Windows는 num_workers=0 유지, collate_fn만 개선된 버전 사용)
train_loader = DataLoader(
    train_ds, batch_size=1, shuffle=True,
    collate_fn=DataCollator(processor, True),
    num_workers=0   # Windows에서 num_workers>0은 에러 발생 가능
)
valid_loader = DataLoader(
    valid_ds, batch_size=1, shuffle=False,
    collate_fn=DataCollator(processor, True),
    num_workers=0
)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
from tqdm.auto import tqdm

model = model.to(device)
GRAD_ACCUM = 4
SAVE_DIR = "./qwen3_vl_8b_lora"

# 옵티마이저, 학습 스케줄러
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.01)
NUM_EPOCHS = 1  # 이어서 학습할 에포크 수
num_training_steps = NUM_EPOCHS * math.ceil(len(train_loader)/GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(optimizer, int(num_training_steps*0.03), num_training_steps)

# 스케일러
scaler = torch.amp.GradScaler('cuda', enabled=True)

# 학습 루프
global_step = 0

best_val_loss = float("inf")

for epoch in range(NUM_EPOCHS):
    running = 0.0
    model.train()  # ← train 모드 명시 (epoch 시작 시)
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")
    
    for step, batch in enumerate(progress_bar, start=1):  # ← 이 줄이 빠져 있었음
        batch = {k: v.to(device) for k, v in batch.items()}  # ← autocast 밖으로 이동
        
        with torch.amp.autocast('cuda', dtype=torch.bfloat16):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0:
            scaler.unscale_(optimizer)                        # ← gradient clipping 전에 필요
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ← 추가 권장
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1

            avg_loss = running / GRAD_ACCUM
            progress_bar.set_postfix({"loss": f"{avg_loss:.3f}", "step": global_step})
            running = 0.0

    # ── 루프 끝난 뒤 잔여 gradient 처리 ──────────────────────────
    if step % GRAD_ACCUM != 0:  # 마지막 배치가 accumulation 경계가 아닐 때
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()
    model.eval()
    val_loss = 0.0
    val_steps = 0
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=torch.bfloat16):
        for vb in tqdm(valid_loader, desc=f"Epoch {epoch+1} [valid]", unit="batch"):
            vb = {k: v.to(device) for k, v in vb.items()}
            val_loss += model(**vb).loss.item()
            val_steps += 1
    avg_val = val_loss / val_steps
    print(f"[Epoch {epoch+1}] valid loss {avg_val:.4f}")
    if avg_val < best_val_loss:
        best_val_loss = avg_val
        model.save_pretrained(SAVE_DIR)
        processor.save_pretrained(SAVE_DIR)
        print(f"  ✓ Best model saved (loss: {best_val_loss:.4f})")


Epoch 1 [train]:   0%|          | 0/180 [00:00<?, ?batch/s]c:\SSAFY\2026-ssafy-ai-15-2\baseline\Lib\site-packages\torch\utils\checkpoint.py:235: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
Epoch 1 [train]:  10%|█         | 18/180 [12:38<1:52:51, 41.80s/batch, loss=0.000, step=4]

In [ ]:
import os
print(os.listdir(SAVE_DIR))

['adapter_config.json', 'adapter_model.safetensors', 'added_tokens.json', 'chat_template.jinja', 'merges.txt', 'preprocessor_config.json', 'README.md', 'special_tokens_map.json', 'tokenizer.json', 'tokenizer_config.json', 'video_preprocessor_config.json', 'vocab.json']


# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [ ]:
best_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

SAVE_DIR = "./qwen3_vl_8b_lora"
best_model = PeftModel.from_pretrained(best_model, SAVE_DIR)
best_model.eval()


# ── 선지 파서 ──────────────────────────────────────────
# 코드2의 "마지막 줄 우선" + 코드1의 "answer: X 패턴" 보조 결합
def extract_choice(text: str) -> str:
    text = text.strip().lower()

    # 1순위: 마지막 줄이 단독 선지인 경우 (코드2 방식)
    lines = [l.strip() for l in text.splitlines() if l.strip()]
    if lines:
        last = lines[-1]
        if last in ("a", "b", "c", "d"):
            return last
        tokens = last.split()
        for tok in tokens:
            if tok in ("a", "b", "c", "d"):
                return tok

    # 2순위: "answer: X" 명시 패턴 보조 (코드1 방식)
    m = re.search(r'answer[:\s]+([abcd])', text)
    if m:
        return m.group(1)

    return "a"

# ── 추론 루프 ──────────────────────────────────────────
preds = []

for i in tqdm(range(len(test_df)), desc="Inference", unit="sample"):
    row  = test_df.iloc[i]
    img  = Image.open(row["path"]).convert("RGB")
    user_text = build_mc_prompt(row["question"], row["a"], row["b"], row["c"], row["d"])

    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user",   "content": [
            {"type": "image", "image": img},
            {"type": "text",  "text": user_text}
        ]}
    ]

    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=[img], return_tensors="pt").to(device)

    # ▸ AMP + no_grad (코드1) / ▸ best_model 사용 (코드1, 코드2 버그 수정)
    with torch.no_grad(), torch.amp.autocast("cuda", dtype=torch.bfloat16):
        out_ids = best_model.generate(
            **inputs,
            max_new_tokens=4,        # "Answer: A" 형식까지 커버
            do_sample=False,
            temperature=None,        # 명시적 비활성화
            top_p=None,
            eos_token_id=processor.tokenizer.eos_token_id,
        )

    # ▸ 생성분만 디코딩 (코드1) → 프롬프트 내 선지 오파싱 방지
    generated    = out_ids[:, inputs["input_ids"].shape[1]:]
    output_text  = processor.batch_decode(generated, skip_special_tokens=True)[0]
    preds.append(extract_choice(output_text))

    # ▸ VRAM 관리 (코드1)
    del inputs, out_ids, generated
    torch.cuda.empty_cache()


# ── 제출 파일 생성 ─────────────────────────────────────
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("./content/submission.csv", index=False)
print("Saved: ./content/submission.csv")

NameError: name 'AutoModelForImageTextToText' is not defined

In [ ]:
# 모델 응답 예시
print(output_text)

system
You are a helpful visual question answering assistant. Answer using exactly one letter among a, b, c, or d. No explanation.
user
이 사진에 보이는 음식 중 치킨 너겟과 함께 제공된 음식은 무엇인가요?
(a) 피클
(b) 감자튀김
(c) 케첩
(d) 셀러리 스틱과 소스

정답을 반드시 a, b, c, d 중 하나의 소문자 한 글자로만 출력하세요.
assistant
d
